# Comparação de Desempenho: Joey vs PyTorch para MNIST

Este notebook implementa uma rede neural convolucional (LeNet) para o problema MNIST usando tanto Joey quanto PyTorch, e compara o desempenho computacional de ambas as implementações.

## Objetivos
1. Implementar LeNet usando Joey
2. Implementar LeNet usando PyTorch
3. Comparar tempo de execução para:
   - Forward pass
   - Backward pass
   - Training completo
4. Analisar os resultados

## 0. Instalação

Execute esta célula para instalar o Joey e suas dependências:

In [ ]:
# Instalar Joey e dependências
import sys
import subprocess

def install_package(package):
    """Instala um pacote usando pip."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Verificar se Joey está instalado, se não, instalar
try:
    import joey
    print("✓ Joey já está instalado!")
except ImportError:
    print("Joey não encontrado. Instalando...")
    
    # Opção 1: Instalar do repositório GitHub (versão de desenvolvimento)
    # install_package("git+https://github.com/devitocodes/joey.git")
    
    # Opção 2: Instalar de um clone local (se você clonou o repositório)
    # Descomente e ajuste o caminho se necessário:
    # install_package("-e /caminho/para/joey")
    
    # Opção 3: Instalar dependências manualmente
    print("Instalando dependências do Joey...")
    install_package("devito>=4.8.0")
    install_package("torch>=2.0.0")
    install_package("numpy>=1.20.0")
    
    print("\n⚠️  IMPORTANTE:")
    print("Para usar Joey, você precisa instalar o pacote completo.")
    print("Execute uma das seguintes opções:\n")
    print("1. Instalar do GitHub:")
    print("   pip install git+https://github.com/devitocodes/joey.git\n")
    print("2. Clonar e instalar localmente:")
    print("   git clone https://github.com/devitocodes/joey.git")
    print("   pip install -e joey\n")
    print("Após instalar, reinicie o kernel do Jupyter e execute novamente.")

# Instalar outras dependências necessárias para o notebook
packages = [
    "torchvision>=0.15.0",
    "matplotlib",
]

print("\nVerificando outras dependências...")
for package in packages:
    try:
        pkg_name = package.split(">=")[0].split("==")[0]
        __import__(pkg_name)
        print(f"✓ {pkg_name} já está instalado")
    except ImportError:
        print(f"Instalando {package}...")
        install_package(package)

print("\n✅ Instalação concluída!")

## 1. Importações e Configuração

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import joey
from joey.activation import ReLU
import time
import matplotlib.pyplot as plt
from devito import logger

# Configurar logging do Devito para não poluir a saída
logger.set_log_noperf()

# Configurações
BATCH_SIZE = 4
NUM_WORKERS = 2
SEED = 42

# Definir seed para reprodutibilidade
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Importações concluídas!")
print(f"PyTorch version: {torch.__version__}")
print(f"Batch size: {BATCH_SIZE}")

## 2. Carregar Dataset MNIST

In [ ]:
# Transformações para normalização
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # LeNet espera imagens 32x32
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5)
])

# Download e carregamento do dataset
trainset = torchvision.datasets.MNIST(
    root='./mnist',
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.MNIST(
    root='./mnist',
    train=False,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(f"Dataset carregado!")
print(f"Training samples: {len(trainset)}")
print(f"Test samples: {len(testset)}")

## 3. Definir Arquitetura LeNet

### 3.1 Versão PyTorch

In [ ]:
class PyTorchLeNet(nn.Module):
    """LeNet-5 em PyTorch para MNIST."""
    
    def __init__(self):
        super(PyTorchLeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 3)      # 1 canal entrada, 6 filtros 3x3
        self.conv2 = nn.Conv2d(6, 16, 3)     # 6 canais entrada, 16 filtros 3x3
        self.fc1 = nn.Linear(16 * 6 * 6, 120)  # Camada totalmente conectada
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)         # 10 classes (dígitos 0-9)
    
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
    def num_flat_features(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

print("Arquitetura PyTorch LeNet definida!")

### 3.2 Versão Joey

In [ ]:
def create_joey_lenet(batch_size=4):
    """Cria LeNet-5 usando Joey."""
    
    # Camada 1: Convolução 6 filtros 3x3 + ReLU
    layer1 = joey.Conv(
        kernel_size=(6, 3, 3),
        input_size=(batch_size, 1, 32, 32),
        activation=ReLU(),
        generate_code=False
    )
    
    # Camada 2: Max pooling 2x2
    layer2 = joey.MaxPooling(
        kernel_size=(2, 2),
        input_size=(batch_size, 6, 30, 30),
        stride=(2, 2),
        generate_code=False
    )
    
    # Camada 3: Convolução 16 filtros 3x3 + ReLU
    layer3 = joey.Conv(
        kernel_size=(16, 3, 3),
        input_size=(batch_size, 6, 15, 15),
        activation=ReLU(),
        generate_code=False
    )
    
    # Camada 4: Max pooling 2x2
    layer4 = joey.MaxPooling(
        kernel_size=(2, 2),
        input_size=(batch_size, 16, 13, 13),
        stride=(2, 2),
        strict_stride_check=False,
        generate_code=False
    )
    
    # Camada de achatamento
    layer_flat = joey.Flat(
        input_size=(batch_size, 16, 6, 6),
        generate_code=False
    )
    
    # Camada 5: Fully Connected (576 -> 120) + ReLU
    layer5 = joey.FullyConnected(
        weight_size=(120, 576),
        input_size=(576, batch_size),
        activation=ReLU(),
        generate_code=False
    )
    
    # Camada 6: Fully Connected (120 -> 84) + ReLU
    layer6 = joey.FullyConnected(
        weight_size=(84, 120),
        input_size=(120, batch_size),
        activation=ReLU(),
        generate_code=False
    )
    
    # Camada 7: Fully Connected (84 -> 10) - Saída
    layer7 = joey.FullyConnected(
        weight_size=(10, 84),
        input_size=(84, batch_size),
        generate_code=False
    )
    
    # Criar rede
    layers = [layer1, layer2, layer3, layer4, layer_flat, layer5, layer6, layer7]
    net = joey.Net(layers)
    
    return net, layers

print("Função para criar Joey LeNet definida!")

## 4. Inicializar Redes com Mesmos Pesos

Para uma comparação justa, vamos inicializar ambas as redes com os mesmos pesos.

In [ ]:
# Criar redes
joey_net, joey_layers = create_joey_lenet(BATCH_SIZE)
pytorch_net = PyTorchLeNet()
pytorch_net.double()  # Joey trabalha com float64

# Copiar pesos do Joey para PyTorch para garantir mesmos valores iniciais
with torch.no_grad():
    pytorch_net.conv1.weight[:] = torch.from_numpy(joey_layers[0].kernel.data)
    pytorch_net.conv1.bias[:] = torch.from_numpy(joey_layers[0].bias.data)
    
    pytorch_net.conv2.weight[:] = torch.from_numpy(joey_layers[2].kernel.data)
    pytorch_net.conv2.bias[:] = torch.from_numpy(joey_layers[2].bias.data)
    
    pytorch_net.fc1.weight[:] = torch.from_numpy(joey_layers[5].kernel.data)
    pytorch_net.fc1.bias[:] = torch.from_numpy(joey_layers[5].bias.data)
    
    pytorch_net.fc2.weight[:] = torch.from_numpy(joey_layers[6].kernel.data)
    pytorch_net.fc2.bias[:] = torch.from_numpy(joey_layers[6].bias.data)
    
    pytorch_net.fc3.weight[:] = torch.from_numpy(joey_layers[7].kernel.data)
    pytorch_net.fc3.bias[:] = torch.from_numpy(joey_layers[7].bias.data)

print("Redes inicializadas com os mesmos pesos!")

## 5. Benchmark: Forward Pass

Vamos medir o tempo de execução do forward pass para ambas as implementações.

In [ ]:
# Obter um batch de teste
test_images, test_labels = next(iter(testloader))
test_images_np = test_images.double().numpy()

NUM_RUNS = 10

# Benchmark Joey - Forward Pass
joey_forward_times = []
print(f"\nExecutando {NUM_RUNS} forward passes com Joey...")
for i in range(NUM_RUNS):
    start_time = time.time()
    joey_output = joey_net.forward(test_images_np)
    end_time = time.time()
    joey_forward_times.append(end_time - start_time)
    if i == 0:
        # Salvar primeira saída para comparação
        joey_first_output = joey_layers[7].result.data.copy()

joey_avg_forward = np.mean(joey_forward_times)
joey_std_forward = np.std(joey_forward_times)

# Benchmark PyTorch - Forward Pass
pytorch_forward_times = []
print(f"Executando {NUM_RUNS} forward passes com PyTorch...")
for i in range(NUM_RUNS):
    start_time = time.time()
    pytorch_output = pytorch_net(test_images.double())
    end_time = time.time()
    pytorch_forward_times.append(end_time - start_time)
    if i == 0:
        # Salvar primeira saída para comparação
        pytorch_first_output = pytorch_output.detach().numpy().T

pytorch_avg_forward = np.mean(pytorch_forward_times)
pytorch_std_forward = np.std(pytorch_forward_times)

# Calcular erro relativo entre as saídas
relative_error = np.abs(joey_first_output - pytorch_first_output) / (np.abs(pytorch_first_output) + 1e-10)
max_error = np.nanmax(relative_error)

print("\n" + "="*60)
print("RESULTADOS - FORWARD PASS")
print("="*60)
print(f"\nJoey:")
print(f"  Tempo médio: {joey_avg_forward*1000:.3f} ms ± {joey_std_forward*1000:.3f} ms")
print(f"\nPyTorch:")
print(f"  Tempo médio: {pytorch_avg_forward*1000:.3f} ms ± {pytorch_std_forward*1000:.3f} ms")
print(f"\nSpeedup: {joey_avg_forward/pytorch_avg_forward:.2f}x")
if joey_avg_forward < pytorch_avg_forward:
    print(f"Joey é {pytorch_avg_forward/joey_avg_forward:.2f}x mais rápido")
else:
    print(f"PyTorch é {joey_avg_forward/pytorch_avg_forward:.2f}x mais rápido")
print(f"\nErro relativo máximo entre saídas: {max_error:.2e}")
print("="*60)

## 6. Benchmark: Backward Pass

Agora vamos medir o tempo do backward pass (cálculo de gradientes).

In [ ]:
# Função de perda para Joey (gradiente manual)
def joey_loss_grad(output_layer, expected_labels):
    """Calcula gradientes para cross-entropy loss no Joey."""
    gradients = []
    for b in range(BATCH_SIZE):
        row = []
        for j in range(10):
            result = output_layer.result.data[j, b]
            if j == expected_labels[b]:
                result -= 1
            row.append(result)
        gradients.append(row)
    return gradients

# Benchmark Joey - Backward Pass
joey_backward_times = []
print(f"\nExecutando {NUM_RUNS} backward passes com Joey...")
for i in range(NUM_RUNS):
    # Forward pass primeiro
    joey_net.forward(test_images_np)
    
    # Medir backward pass
    start_time = time.time()
    joey_net.backward(test_labels.numpy(), joey_loss_grad)
    end_time = time.time()
    joey_backward_times.append(end_time - start_time)

joey_avg_backward = np.mean(joey_backward_times)
joey_std_backward = np.std(joey_backward_times)

# Benchmark PyTorch - Backward Pass
criterion = nn.CrossEntropyLoss()
pytorch_backward_times = []
print(f"Executando {NUM_RUNS} backward passes com PyTorch...")
for i in range(NUM_RUNS):
    # Forward pass primeiro
    pytorch_net.zero_grad()
    outputs = pytorch_net(test_images.double())
    loss = criterion(outputs, test_labels)
    
    # Medir backward pass
    start_time = time.time()
    loss.backward()
    end_time = time.time()
    pytorch_backward_times.append(end_time - start_time)

pytorch_avg_backward = np.mean(pytorch_backward_times)
pytorch_std_backward = np.std(pytorch_backward_times)

print("\n" + "="*60)
print("RESULTADOS - BACKWARD PASS")
print("="*60)
print(f"\nJoey:")
print(f"  Tempo médio: {joey_avg_backward*1000:.3f} ms ± {joey_std_backward*1000:.3f} ms")
print(f"\nPyTorch:")
print(f"  Tempo médio: {pytorch_avg_backward*1000:.3f} ms ± {pytorch_std_backward*1000:.3f} ms")
print(f"\nSpeedup: {joey_avg_backward/pytorch_avg_backward:.2f}x")
if joey_avg_backward < pytorch_avg_backward:
    print(f"Joey é {pytorch_avg_backward/joey_avg_backward:.2f}x mais rápido")
else:
    print(f"PyTorch é {joey_avg_backward/pytorch_avg_backward:.2f}x mais rápido")
print("="*60)

## 7. Benchmark: Training Loop Completo

Vamos executar um loop de treinamento completo com múltiplos batches.

In [ ]:
NUM_BATCHES = 20  # Número de batches para treinar
LEARNING_RATE = 0.001
MOMENTUM = 0.9

# Reinicializar redes para treinamento justo
joey_net, joey_layers = create_joey_lenet(BATCH_SIZE)
pytorch_net = PyTorchLeNet()
pytorch_net.double()

# Sincronizar pesos iniciais
with torch.no_grad():
    pytorch_net.conv1.weight[:] = torch.from_numpy(joey_layers[0].kernel.data)
    pytorch_net.conv1.bias[:] = torch.from_numpy(joey_layers[0].bias.data)
    pytorch_net.conv2.weight[:] = torch.from_numpy(joey_layers[2].kernel.data)
    pytorch_net.conv2.bias[:] = torch.from_numpy(joey_layers[2].bias.data)
    pytorch_net.fc1.weight[:] = torch.from_numpy(joey_layers[5].kernel.data)
    pytorch_net.fc1.bias[:] = torch.from_numpy(joey_layers[5].bias.data)
    pytorch_net.fc2.weight[:] = torch.from_numpy(joey_layers[6].kernel.data)
    pytorch_net.fc2.bias[:] = torch.from_numpy(joey_layers[6].bias.data)
    pytorch_net.fc3.weight[:] = torch.from_numpy(joey_layers[7].kernel.data)
    pytorch_net.fc3.bias[:] = torch.from_numpy(joey_layers[7].bias.data)

# Training Joey
print(f"\nTreinando Joey por {NUM_BATCHES} batches...")
joey_optimizer = optim.SGD(joey_net.pytorch_parameters, lr=LEARNING_RATE, momentum=MOMENTUM)

joey_start = time.time()
batch_count = 0
for images, labels in trainloader:
    if batch_count >= NUM_BATCHES:
        break
    
    images_np = images.double().numpy()
    joey_net.forward(images_np)
    joey_net.backward(labels.numpy(), joey_loss_grad, joey_optimizer)
    batch_count += 1

joey_training_time = time.time() - joey_start

# Training PyTorch
print(f"Treinando PyTorch por {NUM_BATCHES} batches...")
pytorch_optimizer = optim.SGD(pytorch_net.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()

pytorch_start = time.time()
batch_count = 0
for images, labels in trainloader:
    if batch_count >= NUM_BATCHES:
        break
    
    pytorch_optimizer.zero_grad()
    outputs = pytorch_net(images.double())
    loss = criterion(outputs, labels)
    loss.backward()
    pytorch_optimizer.step()
    batch_count += 1

pytorch_training_time = time.time() - pytorch_start

print("\n" + "="*60)
print("RESULTADOS - TRAINING COMPLETO")
print("="*60)
print(f"\nJoey:")
print(f"  Tempo total: {joey_training_time:.3f} s")
print(f"  Tempo por batch: {joey_training_time/NUM_BATCHES*1000:.3f} ms")
print(f"\nPyTorch:")
print(f"  Tempo total: {pytorch_training_time:.3f} s")
print(f"  Tempo por batch: {pytorch_training_time/NUM_BATCHES*1000:.3f} ms")
print(f"\nSpeedup: {joey_training_time/pytorch_training_time:.2f}x")
if joey_training_time < pytorch_training_time:
    print(f"Joey é {pytorch_training_time/joey_training_time:.2f}x mais rápido")
else:
    print(f"PyTorch é {joey_training_time/pytorch_training_time:.2f}x mais rápido")
print("="*60)

## 8. Visualização dos Resultados

In [ ]:
# Criar gráficos de comparação
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico 1: Forward Pass
ax1 = axes[0]
x_pos = [0, 1]
forward_means = [joey_avg_forward*1000, pytorch_avg_forward*1000]
forward_stds = [joey_std_forward*1000, pytorch_std_forward*1000]
colors = ['#3498db', '#e74c3c']
ax1.bar(x_pos, forward_means, yerr=forward_stds, color=colors, alpha=0.7, capsize=5)
ax1.set_ylabel('Tempo (ms)', fontsize=12)
ax1.set_title('Forward Pass', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(['Joey', 'PyTorch'], fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Adicionar valores no topo das barras
for i, (mean, std) in enumerate(zip(forward_means, forward_stds)):
    ax1.text(i, mean + std + 0.5, f'{mean:.2f}±{std:.2f}', 
             ha='center', va='bottom', fontsize=9)

# Gráfico 2: Backward Pass
ax2 = axes[1]
backward_means = [joey_avg_backward*1000, pytorch_avg_backward*1000]
backward_stds = [joey_std_backward*1000, pytorch_std_backward*1000]
ax2.bar(x_pos, backward_means, yerr=backward_stds, color=colors, alpha=0.7, capsize=5)
ax2.set_ylabel('Tempo (ms)', fontsize=12)
ax2.set_title('Backward Pass', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['Joey', 'PyTorch'], fontsize=11)
ax2.grid(axis='y', alpha=0.3)

for i, (mean, std) in enumerate(zip(backward_means, backward_stds)):
    ax2.text(i, mean + std + 0.5, f'{mean:.2f}±{std:.2f}', 
             ha='center', va='bottom', fontsize=9)

# Gráfico 3: Training Completo
ax3 = axes[2]
training_times = [joey_training_time, pytorch_training_time]
ax3.bar(x_pos, training_times, color=colors, alpha=0.7)
ax3.set_ylabel('Tempo (s)', fontsize=12)
ax3.set_title(f'Training ({NUM_BATCHES} batches)', fontsize=14, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(['Joey', 'PyTorch'], fontsize=11)
ax3.grid(axis='y', alpha=0.3)

for i, time_val in enumerate(training_times):
    ax3.text(i, time_val + 0.1, f'{time_val:.2f}s', 
             ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('joey_vs_pytorch_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Gráfico salvo como 'joey_vs_pytorch_performance.png'")

## 9. Resumo Final

In [ ]:
print("\n" + "="*70)
print(" "*20 + "RESUMO COMPARATIVO FINAL")
print("="*70)
print("\nOperação          | Joey (ms)      | PyTorch (ms)   | Speedup")
print("-"*70)
print(f"Forward Pass      | {joey_avg_forward*1000:>10.3f}     | {pytorch_avg_forward*1000:>10.3f}     | {joey_avg_forward/pytorch_avg_forward:>6.2f}x")
print(f"Backward Pass     | {joey_avg_backward*1000:>10.3f}     | {pytorch_avg_backward*1000:>10.3f}     | {joey_avg_backward/pytorch_avg_backward:>6.2f}x")
print(f"Training (total)  | {joey_training_time*1000:>10.3f}     | {pytorch_training_time*1000:>10.3f}     | {joey_training_time/pytorch_training_time:>6.2f}x")
print("="*70)
print("\nCONCLUSÕES:")
print("-"*70)

if joey_avg_forward < pytorch_avg_forward:
    print(f"✓ Joey é {pytorch_avg_forward/joey_avg_forward:.2f}x mais rápido no forward pass")
else:
    print(f"✗ PyTorch é {joey_avg_forward/pytorch_avg_forward:.2f}x mais rápido no forward pass")

if joey_avg_backward < pytorch_avg_backward:
    print(f"✓ Joey é {pytorch_avg_backward/joey_avg_backward:.2f}x mais rápido no backward pass")
else:
    print(f"✗ PyTorch é {joey_avg_backward/pytorch_avg_backward:.2f}x mais rápido no backward pass")

if joey_training_time < pytorch_training_time:
    print(f"✓ Joey é {pytorch_training_time/joey_training_time:.2f}x mais rápido no training completo")
else:
    print(f"✗ PyTorch é {joey_training_time/pytorch_training_time:.2f}x mais rápido no training completo")

print(f"\n✓ Erro numérico máximo: {max_error:.2e} (resultados numericamente equivalentes)")
print("\nNOTA: Joey gera código otimizado em tempo real usando Devito,")
print("      enquanto PyTorch usa operações pré-compiladas e otimizadas.")
print("      O desempenho pode variar dependendo da arquitetura da rede,")
print("      tamanho do batch e hardware utilizado.")
print("="*70)